# Quality assessment

In [50]:
import rasterio
import os
import numpy as np
from tqdm import tqdm
import pickle
import shutil
from PIL import Image

In [74]:
src_gt_folder = r"D:\GitHubProjects\Terranum_repo\LandSlides\segformerlandslides\notebooks\data_for_testing\images\gt\gt_bin"
src_preds_folder = r"D:\GitHubProjects\Terranum_repo\LandSlides\segformerlandslides\notebooks\data_for_testing\images\predictions\preds_bin"

In [75]:
lst_gt_files = [x for x in os.listdir(src_gt_folder) if x.endswith('.tif')]
lst_preds_files = [x.replace('_mask', '') for x in os.listdir(src_preds_folder) if x.endswith('.tif')]

assert lst_gt_files == lst_preds_files

In [80]:
num_found = 0
lst_not_found = []
for id_img, img in tqdm(enumerate(lst_gt_files), total=len(lst_gt_files)):
    src_gt_image = os.path.join(src_gt_folder, img)
    src_pred_image = os.path.join(src_preds_folder, img.replace('.tif', '_mask.tif'))
    gt_arr = rasterio.open(src_gt_image).read()[0,...]
    gt_arr[gt_arr == 255] = 1
    preds_arr = rasterio.open(src_pred_image).read().squeeze(0)
    
    # if "image_relation_1299821" in img:
    #     print(set(list(gt_arr.flatten())))
    #     print(set(list(preds_arr.flatten())))
    gt_arr[gt_arr == 0] = 4
    preds_arr[preds_arr == 0] = 5

    matching_mask = gt_arr == preds_arr
    # if np.sum(matching_mask) > 0 or len(set(list(gt_arr.flatten()))) == 1:
    if np.sum(matching_mask) > 0:
        num_found += 1
    else:
        lst_not_found.append(src_gt_image)
    # if "image_relation_1299821" in img:
    #     print(np.sum(gt_arr))
    #     print(np.sum(preds_arr))
    #     print(np.sum(gt_arr == preds_arr))
    #     break
print("Num found: ", num_found)
print("Frac found: ", round(num_found / len(lst_gt_files)*100, 2), '%')

  0%|          | 0/1005 [00:00<?, ?it/s]d:\GitHubProjects\Terranum_repo\LandSlides\segformerlandslides\.venv\lib\site-packages\rasterio\__init__.py:387: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)
100%|██████████| 1005/1005 [03:40<00:00,  4.57it/s]

Num found:  947
Frac found:  94.23 %


In [ ]:
# copy the failed ones to a separate folder
src_folder_failed = 

494


### Select sampels that are not in training set

In [4]:
with open(r"D:\GitHubProjects\Terranum_repo\LandSlides\segformerlandslides\notebooks\list_images_training.pickle", 'rb') as f:
    lst_training = pickle.load(f)
print(lst_training[0])
lst_clean = [os.path.basename(x).split('_scale')[0] + '.tif' for x in lst_training]
print(lst_clean)

data/dataset_segmentation_playground\dataset_playground_multires_6000\images\image_relation_10440641_scale_0.25_0.tif
['image_relation_10440641.tif', 'image_relation_10440641.tif', 'image_relation_10440641.tif', 'image_relation_10440641.tif', 'image_relation_12138209.tif', 'image_relation_12138209.tif', 'image_relation_12138209.tif', 'image_relation_12138209.tif', 'image_relation_12648860.tif', 'image_relation_12648860.tif', 'image_relation_12648860.tif', 'image_relation_12648860.tif', 'image_relation_1271933.tif', 'image_relation_1271933.tif', 'image_relation_1271933.tif', 'image_relation_1271933.tif', 'image_relation_13154653.tif', 'image_relation_13154653.tif', 'image_relation_13154653.tif', 'image_relation_13154653.tif', 'image_relation_14696347.tif', 'image_relation_14696347.tif', 'image_relation_14696347.tif', 'image_relation_14696347.tif', 'image_relation_1589976.tif', 'image_relation_1589976.tif', 'image_relation_1589976.tif', 'image_relation_1589976.tif', 'image_relation_16002

In [6]:
src_data_1 = r"E:\Terranum_FATTY\Segformer\tiles\Playgrounds\4100_tiles\dataset_1\images"
src_data_2 = r"E:\Terranum_FATTY\Segformer\tiles\Playgrounds\4100_tiles\dataset_2\images"

lst_data = [os.path.join(src_data_1, x) for x in os.listdir(src_data_1)]
for x in os.listdir(src_data_2):
    lst_data.append(os.path.join(src_data_2, x))

In [7]:
print(len(lst_data))

6567


In [8]:
lst_not_in_training = []
for x in lst_data:
    if os.path.basename(x) not in lst_clean:
        lst_not_in_training.append(x)

In [9]:
print(len(lst_not_in_training))

2600


In [12]:
src_results = r"D:\GitHubProjects\Terranum_repo\LandSlides\segformerlandslides\notebooks\data_for_testing"
src_res_images = os.path.join(src_results, 'images')
src_res_gt = os.path.join(src_results, 'gt')
os.makedirs(src_res_images, exist_ok=True)
os.makedirs(src_res_gt, exist_ok=True)

for _, x in tqdm(enumerate(lst_not_in_training), total=len(lst_not_in_training), desc="Copying images"):
    shutil.copyfile(
        x,
        os.path.join(src_res_images, os.path.basename(x))
    )
    
    x_bin = os.path.join(os.path.dirname(os.path.dirname(x)), 'masks', os.path.basename(x))
    shutil.copyfile(
        x_bin,
        os.path.join(src_res_gt, os.path.basename(x))
    )


Copying images: 100%|██████████| 2600/2600 [12:41<00:00,  3.41it/s]


In [20]:
list_samples = [x.replace('_mask', '') for x in os.listdir(r"D:\GitHubProjects\Terranum_repo\LandSlides\segformerlandslides\notebooks\data_for_testing\images\test\preds_bin")]
for x in list_samples:
    shutil.copyfile(
        os.path.join(r"D:\GitHubProjects\Terranum_repo\LandSlides\segformerlandslides\notebooks\data_for_testing\gt", x),
        os.path.join(r"D:\GitHubProjects\Terranum_repo\LandSlides\segformerlandslides\notebooks\data_for_testing\images\test\gt", x)
    )


In [49]:
from rasterio.transform import from_origin
src = r"D:\GitHubProjects\Terranum_repo\LandSlides\segformerlandslides\notebooks\data_for_testing\images\test\preds_img"
src_dest = r"D:\GitHubProjects\Terranum_repo\LandSlides\segformerlandslides\notebooks\data_for_testing\images\test\test"
lst_files = [x for x in os.listdir(src)]
for _, x in tqdm(enumerate(lst_files), total=len(lst_files)):
    with rasterio.open(os.path.join(src, x)) as f:
        data = f.read()
        # transform = f.transform
        profile = f.profile
    # print(data.shape)
    # # Flip the data vertically and fix the transform
    # import numpy as np
    # data_flipped = np.flip(data, axis=1)  # flip along rows

    # # Rebuild transform with -1 Y pixel size
    # new_transform = from_origin(
    #     transform.c,                    # left x
    #     transform.f + transform.e * data.shape[1],  # new top y
    #     transform.a,                    # pixel width
    #     abs(transform.e)                # pixel height (positive → negative internally)
    # )

    # profile.update(transform=new_transform)
    
    # with rasterio.open(os.path.join(src_dest, x), "w", **profile) as dst:
    
    # with rasterio.open(os.path.join(src_dest, x), "w") as dst:
        # dst.write(data)
    Image.fromarray(np.moveaxis(data, 0, 2)).save(os.path.join(src_dest, x))
    # break


100%|██████████| 100/100 [00:50<00:00,  1.98it/s]


In [55]:
src_preds = r"D:\GitHubProjects\Terranum_repo\LandSlides\segformerlandslides\notebooks\data_for_testing\images\predictions\preds_bin"
src_images_src = r"D:\GitHubProjects\Terranum_repo\LandSlides\segformerlandslides\notebooks\data_for_testing\gt"
src_images_dest = r"D:\GitHubProjects\Terranum_repo\LandSlides\segformerlandslides\notebooks\data_for_testing\images\gt"

for _, file in tqdm(enumerate(os.listdir(src_preds)), total=len(os.listdir(src_preds))):
    if file.replace('_mask','') in os.listdir(src_images_src):
        os.rename(
            os.path.join(src_images_src, file.replace('_mask', '')),
            os.path.join(src_images_dest, file.replace('_mask', '')),
        )

100%|██████████| 1005/1005 [00:01<00:00, 996.07it/s]


In [73]:
src_gt_img = r"D:\GitHubProjects\Terranum_repo\LandSlides\segformerlandslides\notebooks\data_for_testing\images\gt\gt_img"
src_gt_bin = r"D:\GitHubProjects\Terranum_repo\LandSlides\segformerlandslides\notebooks\data_for_testing\images\gt\gt_bin"

for _, img in tqdm(enumerate(os.listdir(src_gt_img)), total=len(os.listdir(src_gt_img))):
    img_arr = rasterio.open(os.path.join(src_gt_img, img)).read()
    mask = img_arr[0,...] > 0
    # print(np.sum(mask))
    arr_bin = np.zeros((4100, 4100))
    arr_bin[mask] = 1
    # print(np.sum(arr_bin))
    Image.fromarray(arr_bin).save(os.path.join(src_gt_bin, img))
    # break


100%|██████████| 1005/1005 [05:10<00:00,  3.24it/s]


In [62]:
arr = np.zeros((4100,4100))
arr[2000:2100, 2000:2100] = 1
src = r"D:\GitHubProjects\Terranum_repo\LandSlides\segformerlandslides\notebooks\data_for_testing\images\predictions\image_relation_1299821.tif"
Image.fromarray(arr).save(src)